In [1]:
import os
import math
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from skimage.measure import label, regionprops, regionprops_table
from skimage.transform import rotate
from sklearn.cluster import KMeans
from cellpose import denoise, io
import umap
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import pickle
from cellpose import transforms, plot
import csv
import cv2
from skimage.filters import sobel
from skimage.segmentation import expand_labels, watershed
import skimage.segmentation as segmentation
from collections import defaultdict
from pathlib import Path
import multiprocessing
import logging

2026-10-08 11:19:35.603095: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-08 11:19:35.640247: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX512F AVX512_VNNI, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
#### This notebook shows how the masks are created which are then passed to feature extraction, this is demonstrated above
### Before its parallelized, it also tests for corrupted files

In [2]:
def get_binary_from_mask(mask):
    # Convert input to an 8-bit grayscale image
    core_img_grey = (mask // 256).astype(np.uint8)
    
    # Apply binary thresholding 
    _, binary = cv2.threshold(core_img_grey, 1, 1, cv2.THRESH_BINARY)
    
    # Find external contours
    #contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    return binary

In [3]:
import multiprocessing
import torch
num_cpus = multiprocessing.cpu_count()
print(f"You have {num_cpus} CPUs")

if torch.cuda.is_available(): 
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("No GPUs detected")

You have 64 CPUs
Number of GPUs: 1
GPU 0: Tesla V100-SXM2-32GB


In [4]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [PID %(process)d] %(message)s",
)

In [ ]:
# Paths to your directories
overlap_dir = "./ALL_overlap_expanded_results"
segmented_dir = "./segmented_1000"

# Count files in each directory
overlap_files = [f for f in os.listdir(overlap_dir) if f.endswith(".npy")]
segmented_files = [f for f in os.listdir(segmented_dir) if f.endswith(".npy")]

print(f"Number of unique TMAs in ALL_overlap_expanded_results: {len(overlap_files)/5}")
print(f"Number of unique TMAs in segmented_1000: {len(segmented_files)/2}")
print(f"{len(segmented_files)/2 - len(overlap_files)/5}")


In [6]:
class mask_extraction:
    def __init__(self):
        self.cell_body_mask_ls = []

    def process_single_file(self, DAPI_file, Int_Img_File, filename_key, TMA):
        """
        Process a single DAPI mask + intensity image pair.
        Returns a dictionary with results.
        """
        logging.info(f'Here is file running: {TMA}')
        # --- Load the array ---
        DAPI_file = np.load(DAPI_file)
        Int_Img_File = np.load(Int_Img_File)
        
        # --- Masks ---
        expanded_DAPI = expand_labels(DAPI_file, distance=15)
        last_cycle_DAPI = Int_Img_File[5, 0, :, :]
        b_last_cycle_DAPI = get_binary_from_mask(last_cycle_DAPI)
        
        overlap = (DAPI_file & (b_last_cycle_DAPI > 0)).astype(bool)
        
        # --- Watershed ---
        edges = sobel(overlap)
        markers = np.zeros_like(overlap, dtype=int)
        foreground, background = 1, 2
        markers[overlap] = foreground
        markers[~overlap] = background
        ws = watershed(edges, markers)
        seg1 = label(ws == foreground)
        
        expanded_overlap = expand_labels(seg1, distance=15)
        cell_body_mask = (expanded_overlap > 0) & (overlap == 0)
        # --- Counts ---
        first_cycle_cells  = label(DAPI_file).max()
        expanded_DAPI_cells  = label(expanded_DAPI).max()
        last_cycle_cells   = label(b_last_cycle_DAPI).max()
        overlap_cells      = label(overlap).max()
        expanded_cells     = label(expanded_overlap).max()
        
        # --- Pretty Print ---
        print(
            f"  File : {TMA}_{filename_key}\n"
            f"  First-cycle DAPI nuclei cells : {first_cycle_cells}\n"
            f"  First-cycle expanded DAPI nuclei cells : {expanded_DAPI_cells}\n"
            f"  Last-cycle DAPI nuclei cells  : {last_cycle_cells}\n"
            f"  Overlap cells         : {overlap_cells}\n"
            f"  Expanded overlap cells  : {expanded_cells}\n"
        )

        output_dir = "./ALL_overlap_expanded_results"  # Define a central directory to store everything
        os.makedirs(output_dir, exist_ok=True)  # Create the directory if it doesn't exist
        # This prefix ensures TMA is always first
        file_prefix = f"{TMA}_{filename_key}"

        # 3. Save to disk
        np.save(os.path.join(output_dir, f"{file_prefix}_cell_body_mask.npy"), cell_body_mask)
        np.save(os.path.join(output_dir, f"{file_prefix}_expanded_overlap.npy"), expanded_overlap)
        np.save(os.path.join(output_dir, f"{file_prefix}_overlap_mask.npy"), overlap)
        np.save(os.path.join(output_dir, f"{file_prefix}_dapi_masks.npy"), DAPI_file)
        # Slicing the intensity image to exclude DAPI cycle and save space
        np.save(os.path.join(output_dir, f"{file_prefix}_int_img_wo_DAPI.npy"), Int_Img_File[:, 1:, :, :])
        
        # 4. Return METADATA only (No arrays!)
        logging.info(f'TMA: {TMA}, key: {filename_key}, status: Success')
        
        return {
            'filename': filename_key,
            'cell_body_mask': cell_body_mask,
            'expanded_overlap': expanded_overlap,
            'overlap_mask': overlap,
            'DAPI_masks': DAPI_file,
            'Expanded_DAPI_Masks': expanded_DAPI,
            'all_but_DAPI': Int_Img_File[:, 1:, :, :]
        }
        
    def process_batch(self, files_by_number, TMAs_to_include=None, num_workers=None):
        """
        Process all DAPI/intensity files in files_by_number in parallel.
        """
        # Build a list of all (DAPI_file, intensity_file, key) tuples to process

        if num_workers is None:
            num_workers = min(cpu_count(), len(tasks))
        
        chunksize = max(1, len(tasks) // (num_workers * 4))
        
        def process_single_file_wrapper(args):
            return self.process_single_file(*args)

        all_results = []
        with Pool(num_workers) as pool:
            for result in tqdm(
                pool.imap(process_single_file_wrapper, tasks, chunksize=chunksize),
                total=len(tasks),
                desc="Processing images"
            ):
                all_results.append(result)
        self.cell_body_mask_ls = all_results
        return all_results
            